# Main Results

Reproduces Figures 4 through 7 from the report using `data/final_results.csv`.
Run from the `bse/` root directory.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from scipy import stats
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'text.usetex':       False,
    'font.family':       'serif',
    'font.serif':        ['DejaVu Serif'],
    'axes.labelsize':    16,
    'font.size':         11,
    'legend.fontsize':   11,
    'xtick.labelsize':   14,
    'ytick.labelsize':   14,
    'axes.titlesize':    14,
    'figure.dpi':        150,
    'savefig.dpi':       300,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'grid.linestyle':    '--',
    'grid.linewidth':    0.5,
})

COLORS = {
    'ZIP':    '#c0392b', 'PRSH': '#1a3a5c', 'ZIC':  '#27ae60',
    'CUSTOM': '#2980b9', 'SNPR': '#8e44ad', 'SHVR': '#e67e22', 'GVWY': '#95a5a6',
}
SAVE_DIR     = 'figures'
INIT_CASH    = 10000
TRADER_ORDER = ['ZIP', 'CUSTOM', 'PRSH', 'ZIC', 'SNPR', 'SHVR', 'GVWY']
TOP3         = ['ZIP', 'CUSTOM', 'PRSH']
CONDITIONS   = ['stable', 'shock_up', 'shock_down', 'multi_regime']
COND_LABELS  = ['Stable', 'Shock Up', 'Shock Down', 'Multi-Regime']

df = pd.read_csv('data/final_results.csv')
df['profit_norm'] = df['final_profit'] / INIT_CASH
print(f'Loaded {len(df):,} rows | traders: {sorted(df.ttype.unique())} | conditions: {sorted(df.condition.unique())}')

## Summary Table

In [ ]:
summary = (df.groupby('ttype')
             .agg(
                 mean_profit_norm=('profit_norm', 'mean'),
                 std_profit_norm=('profit_norm', 'std'),
                 mean_trades=('n_trades', 'mean'),
                 pct_positive=('final_profit', lambda x: (x > 0).mean() * 100),
             )
             .reindex(TRADER_ORDER)
             .round(4))
summary.columns = ['Mean P/C_0', 'Std P/C_0', 'Mean Trades', '% Profitable']
summary

## Figure 4: Profit Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True, sharey=True)
axes = axes.flatten()

for i, (ax, cond, clabel) in enumerate(zip(axes, CONDITIONS, COND_LABELS)):
    cdf  = df[df['condition'] == cond]
    data = [cdf[cdf['ttype'] == t]['profit_norm'].values for t in TRADER_ORDER]
    bp   = ax.boxplot(data, patch_artist=True, widths=0.55,
                      medianprops=dict(color='black', linewidth=1.5),
                      whiskerprops=dict(linewidth=0.8),
                      capprops=dict(linewidth=0.8),
                      flierprops=dict(marker='o', markersize=2, alpha=0.4, linewidth=0))
    for patch, t in zip(bp['boxes'], TRADER_ORDER):
        patch.set_facecolor(COLORS[t])
        patch.set_alpha(0.85)
    ax.set_xticks(range(1, len(TRADER_ORDER) + 1))
    ax.axhline(0, color='black', linewidth=0.7, linestyle='--', alpha=0.5)
    ax.set_title(clabel)
    if i >= 2:
        ax.set_xticklabels(TRADER_ORDER, fontsize=12, rotation=15)
    if i % 2 == 0:
        ax.set_ylabel(r'$P\,/\,C_0$')

fig.suptitle('Profit Distributions by Market Condition', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'fig4_profit_distributions.png'), dpi=300, bbox_inches='tight')
plt.show()

## Figure 5: Custom Trader Defensive Mode

In [ ]:
inv_df = pd.read_csv('data/inventory_timeseries.csv')

INVENTORY_LIMIT = 294
VOLATILITY_THRESHOLD = 14.987

custom_inv = inv_df[inv_df['ttype'] == 'CUSTOM'].copy()

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True, sharey=True)
axes = axes.flatten()

for ax, cond, label in zip(axes, CONDITIONS, COND_LABELS):
    sub = (custom_inv[custom_inv['condition'] == cond]
           .groupby('timestep')['mean_inv'].mean()
           .reset_index()
           .sort_values('timestep'))
    defensive = (sub['mean_inv'].abs() >= INVENTORY_LIMIT * 0.5).astype(float)
    ax.fill_between(sub['timestep'] * 10, defensive, alpha=0.35,
                    color=COLORS['CUSTOM'], label='Defensive mode')
    ax.plot(sub['timestep'] * 10, defensive,
            color=COLORS['CUSTOM'], linewidth=1.2)
    ax.set_ylim(0, 1.1)
    ax.set_title(label)
    ax.set_xlabel('Time step')
    if ax == axes[0] or ax == axes[2]:
        ax.set_ylabel('Fraction in defensive mode')

fig.suptitle('Custom Trader: Defensive Mode Activation', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'fig5_defensive_mode.png'), dpi=300, bbox_inches='tight')
plt.show()

## Figure 6: Bootstrap Distributions

10,000 bootstrap resamples of mean normalised profit for CT, ZIP and PRSH.

In [ ]:
N_BOOT = 10000
N_SAMP = 150
rng = np.random.default_rng(42)

boot_results = {}
ci_results   = {}
for t in TOP3:
    profits  = df[df['ttype'] == t]['profit_norm'].values
    samples  = rng.choice(profits, size=(N_BOOT, N_SAMP), replace=True)
    means    = samples.mean(axis=1)
    boot_results[t] = means
    ci_results[t]   = (np.percentile(means, 2.5), np.percentile(means, 97.5))

fig, ax = plt.subplots(figsize=(10, 5))
for t in TOP3:
    vals   = boot_results[t]
    lo, hi = ci_results[t]
    ax.hist(vals, bins=80, alpha=0.55, color=COLORS[t], density=True, label=t)
    ax.axvline(lo,          color=COLORS[t], linewidth=1.2, linestyle='--', alpha=0.8)
    ax.axvline(hi,          color=COLORS[t], linewidth=1.2, linestyle='--', alpha=0.8)
    ax.axvline(vals.mean(), color=COLORS[t], linewidth=2.0)

ax.set_xlabel(r'Bootstrapped mean $P\,/\,C_0$')
ax.set_ylabel('Density')
ax.legend(fontsize=12, frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'fig6_bootstrap_distributions.png'), dpi=300, bbox_inches='tight')
plt.show()

print('95% confidence intervals:')
for t in TOP3:
    lo, hi = ci_results[t]
    print(f'  {t}: [{lo:.4f}, {hi:.4f}]')

## Figure 7: Long-Term Cumulative Profit

1,000 bootstrap iterations, each projecting 500 simulated future sessions.

In [ ]:
B     = 1000
N_SIM = 500

fig, ax = plt.subplots(figsize=(11, 6))

for t in TOP3:
    profits = df[df['ttype'] == t]['profit_norm'].values
    draws   = rng.choice(profits, size=(B, N_SIM), replace=True)
    cum     = draws.cumsum(axis=1)
    mean_c  = cum.mean(axis=0)
    lo_c    = np.percentile(cum, 2.5,  axis=0)
    hi_c    = np.percentile(cum, 97.5, axis=0)
    x = np.arange(1, N_SIM + 1)
    ax.plot(x, mean_c, color=COLORS[t], linewidth=2.0, label=t)
    ax.fill_between(x, lo_c, hi_c, color=COLORS[t], alpha=0.25)

ax.set_xlabel('Simulated sessions')
ax.set_ylabel(r'Cumulative $P\,/\,C_0$')
ax.legend(fontsize=12, frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'fig7_cumulative_profit.png'), dpi=300, bbox_inches='tight')
plt.show()

## Bootstrap CI Overlap Check

In [ ]:
print('95% CI summary:')
for t in TOP3:
    lo, hi = ci_results[t]
    print(f'  {t}: [{lo:.4f}, {hi:.4f}]  width = {hi-lo:.4f}')

print()
custom_lo, custom_hi = ci_results['CUSTOM']
for t in ['ZIP', 'PRSH']:
    t_lo, t_hi = ci_results[t]
    overlaps = not (custom_hi < t_lo or t_hi < custom_lo)
    print(f'  CUSTOM vs {t}: overlaps = {overlaps}')